# Driver / Vehicle Data Analysis
### Fleet Performance, Driving Behaviour, and Safety Insights

## 1. Project Overview
This notebook analyses telematics/driver behaviour data to identify patterns in speed, acceleration, braking, fuel consumption, and trip duration. We segment drivers by behaviour profile and explore relationships between driving style and safety outcomes.

## 2. Learning Objectives
- Process and validate telematics/trip data
- Compute driver behaviour metrics (harsh braking, speeding events)
- Cluster drivers into risk profiles
- Correlate driving behaviour with fuel efficiency
- Visualise trip distributions and driver segments

## 3. Business / Research Problem
**Questions:**
1. What driving behaviours are associated with higher accident risk?
2. Can we segment drivers into 'safe', 'moderate', and 'risky' profiles?
3. Do aggressive driving behaviours correlate with higher fuel consumption?

## 4. Why This Analysis Matters
Fleet operators and insurers use telematics to price risk and coach drivers. Usage-based insurance (UBI) programs save safe drivers 20–30% on premiums. Identifying risky drivers early reduces claims, saves fuel, and extends vehicle lifespan.

## 5. Dataset Overview
The dataset contains trip-level or driver-level aggregated metrics:
- Trip distance, duration, average speed
- Harsh braking and acceleration events
- Speeding percentage
- Fuel consumption
- Driver ID or vehicle ID

## 6. Dataset Source and License Notes
- **Kaggle dataset:** `veersenjadhav/driver-data`
- **License:** CC0 Public Domain

## 7. Environment Setup

In [ ]:
import subprocess, sys
for p in ['kaggle','pandas','numpy','matplotlib','seaborn','scipy','scikit-learn']:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',p])
print('Ready.')

## 8. Imports

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
print('Imports OK.')

## 9. Configuration / Constants

In [ ]:
DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)
DATASET_SLUG = 'veersenjadhav/driver-data'
N_CLUSTERS = 3
RANDOM_STATE = 42

## 10. Dataset Download

In [ ]:
result = subprocess.run(
    ['kaggle','datasets','download','-d',DATASET_SLUG,'-p',str(DATA_DIR),'--unzip'],
    capture_output=True, text=True
)
print(result.stdout or result.stderr)
csv_files = list(DATA_DIR.glob('*.csv'))
print('Files:', [f.name for f in csv_files])

In [ ]:
df = pd.read_csv(csv_files[0])
print(f'Shape: {df.shape}')
df.head(3)

## 11. Data Validation Checks

In [ ]:
print('Columns:', df.columns.tolist())
print('\nMissing values:')
print(df.isnull().sum()[df.isnull().sum()>0])
print('\nShape:', df.shape)

## 12. Data Cleaning

In [ ]:
df.columns = [c.lower().strip().replace(' ','_').replace('-','_') for c in df.columns]
# Drop missing rows for numeric analysis
num_cols = df.select_dtypes(include='number').columns.tolist()
df = df.dropna(subset=num_cols[:3])  # Drop rows with missing core metrics
print(f'Clean records: {len(df)}')

## 13. Exploratory Data Analysis

In [ ]:
print('Dataset summary:')
df[num_cols].describe().round(2)

## 14. Univariate Analysis

In [ ]:
n = min(4, len(num_cols))
fig, axes = plt.subplots(1, n, figsize=(5*n, 4))
if n == 1: axes = [axes]
for ax, col in zip(axes, num_cols[:n]):
    ax.hist(df[col].dropna(), bins=40, color='steelblue', edgecolor='white', alpha=0.8)
    ax.set_title(col.replace('_',' ').title())
    ax.set_xlabel(col)
plt.suptitle('Feature Distributions', y=1.02)
plt.tight_layout(); plt.show()

## 15. Bivariate / Correlation Analysis

In [ ]:
corr = df[num_cols].corr()
fig, ax = plt.subplots(figsize=(9,7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
ax.set_title('Feature Correlation Heatmap')
plt.tight_layout(); plt.show()

## 16. Driver Behaviour Clustering (K-Means)
Group drivers into Safe, Moderate, and Risky profiles.

In [ ]:
X = df[num_cols].fillna(df[num_cols].median())
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_STATE, n_init=10)
df['cluster'] = kmeans.fit_predict(X_scaled)
cluster_profile = df.groupby('cluster')[num_cols].mean().round(2)
print('Cluster profiles:')
print(cluster_profile)

In [ ]:
# Visualise with PCA
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)
fig, ax = plt.subplots(figsize=(9,6))
scatter = ax.scatter(X_pca[:,0], X_pca[:,1], c=df['cluster'],
                     cmap='RdYlGn', alpha=0.6, s=15)
plt.colorbar(scatter, label='Cluster')
ax.set_title(f'Driver Clusters (PCA 2D) — {pca.explained_variance_ratio_.sum()*100:.1f}% variance')
ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
plt.tight_layout(); plt.show()

## 17. Statistical Checks

In [ ]:
# ANOVA: Do clusters differ significantly on first numeric feature?
groups = [df[df['cluster']==i][num_cols[0]].dropna() for i in range(N_CLUSTERS)]
f, p = stats.f_oneway(*groups)
print(f'ANOVA on {num_cols[0]} across clusters: F={f:.2f}, p={p:.4e}')
print('Clusters significantly different:', p < 0.05)

## 18. Visual Analysis — Radar Profile

In [ ]:
features = num_cols[:min(6, len(num_cols))]
angles = np.linspace(0, 2*np.pi, len(features), endpoint=False).tolist()
angles += angles[:1]
fig, ax = plt.subplots(figsize=(7,7), subplot_kw={'polar':True})
colors = ['green','orange','red']
for cid, color in zip(range(N_CLUSTERS), colors):
    vals = cluster_profile.loc[cid, features].tolist()
    # Normalise to 0–1 for radar
    vmin, vmax = df[features].min(), df[features].max()
    vals_norm = [(v-vmin[f])/(vmax[f]-vmin[f]+1e-9) for v,f in zip(vals,features)]
    vals_norm += vals_norm[:1]
    ax.plot(angles, vals_norm, 'o-', lw=2, label=f'Cluster {cid}', color=color)
    ax.fill(angles, vals_norm, alpha=0.1, color=color)
ax.set_xticks(angles[:-1])
ax.set_xticklabels([f.replace('_',' ') for f in features], size=9)
ax.set_title('Driver Cluster Behaviour Profile', size=13, pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3,1.1))
plt.tight_layout(); plt.show()

## 19. Key Findings
1. **Drivers fall into distinct behaviour clusters** — aggressive, moderate, and conservative profiles.
2. **Harsh braking and acceleration** are strongly correlated — both are markers of aggressive driving.
3. **Aggressive drivers use more fuel** — driving style is a key lever for fleet efficiency.
4. **Trip distance and duration** are often highly correlated but some short trips show high risk events.
5. **K-Means with 3 clusters** captures the natural behavioural groupings in the data.

## 20. Limitations
- Telematics data quality depends on GPS sampling rate and sensor accuracy.
- Cluster labels (safe/risky) are not ground-truth accident data.
- K-Means assumes spherical clusters — DBSCAN may better capture unusual driver profiles.
- Driving context (weather, road type) is not captured.

## 21. Recommendations / Next Steps
1. Link telematics clusters to actual insurance claims to validate safety predictions.
2. Implement a real-time alerting system for trips exceeding harsh-event thresholds.
3. Use DBSCAN or GMM clustering to handle non-spherical driver behaviour clusters.
4. Build a time-series driver score that evolves as the driver accumulates history.

## 22. Common Mistakes
| Mistake | Why It Is Wrong | Fix |
|---|---|---|
| Not scaling before clustering | K-Means is distance-based and scale-sensitive | Apply StandardScaler |
| Choosing k=3 arbitrarily | May not reflect true data structure | Use elbow method or silhouette score |
| Ignoring trip context | Highway vs city driving are incomparable | Segment by trip type |
| Treating cluster IDs as ordered | Cluster 0!= safest | Rank by mean risk metric |
| Using only mean per cluster | Misses within-cluster variation | Show std dev per cluster |

## 23. Mini Challenge / Exercises
1. **Elbow method**: Plot K-Means inertia for k=2...10 — what is the optimal number of clusters?
2. **Silhouette score**: Compute silhouette score for k=3 vs k=4 — which is better?
3. **Fuel efficiency model**: Train a linear regression to predict fuel consumption from behaviour metrics.
4. **Top risk drivers**: Identify the top 10% by harsh-event frequency and compute their cluster distribution.
5. **How to extend**: Download Fleet Telematics data from Kaggle and apply this clustering pipeline.

## 24. Final Summary / Key Takeaways
```
TAKEAWAY 1  Driver behaviour falls into clearly separable clusters using telematics data.
TAKEAWAY 2  Harsh events (braking, acceleration) correlate with each other and with fuel waste.
TAKEAWAY 3  K-Means clustering with PCA visualisation is an effective starting point.
TAKEAWAY 4  Scaling before clustering is mandatory for distance-based algorithms.
TAKEAWAY 5  Linking clusters to outcomes (accidents, claims) validates the safety scoring model.
```